# Busqueda Precisa y Guiada

Por busqueda guiada entiendase por tener una solución minimal en la ecuación de la recta

![descriptive graph of minimum loss function in search space, in this case a previous search space well behaved in a $dx_i$ of $~ 10^{-7} GeV$ masses](./img/dx_space_m12_mphi.png)

Aprovechando los puntos optimos obtenidos con la busqueda bayesiana es posible tener una funcion para estimar un area plausbible de $m_{12}$

$$
m_{12}^2 = 9.653 \times 10^{-8} x^2 + 1.301 \times 10^{-6} x - 9.184 \times 10^{-5}
$$

donde $x=m_\phi$
Se probo dependencia con respecto a $m_A$ pero en el rango $m_\phi< m_A \in [300, 500]$ no se encontro particularme un cambio significativo de la expresion algebraica obetenida con solo utilizar $m_{12} = m_{12}(m_\phi)$ para los valores fijos utilizados


In [20]:
import numpy as np
import matplotlib.pyplot as plt

from lib.oracle import run_oracle, safe_run_oracle
from lib.utils import estimate_exploration_time, expand_parameter_space
import numpy as np
import matplotlib.pyplot as plt

# Parámetros globales del modelo
lambda6 = 0.1
lambda7 = 0.0
#alpha = 1e-7
#beta  = 1.570796226794897

tanbeta = 1e7
sin_ba = 1

def physpoint(mphi: float, m12_2: float, mA: float = 300.0):
    # Llama al modelo con valores fijos y retorna el diccionario de resultados
    return safe_run_oracle([mphi, mA, sin_ba, tanbeta, lambda6, lambda7, m12_2])

def loss(mphi: float, m12_2: float, mA: float = 300.0):
    # Calcula una métrica basada en la suma de |lambda_i|
    output = physpoint(mphi, m12_2, mA)
    if output is None:
        return 1e14  # Penalización para errores o puntos no físicos
    return np.log(sum(np.abs(output[f'lambda{i}']) for i in range(1,6)) )


# Define función para verificar validez física del modelo
def is_valid(output: dict) -> bool:
    if output is None:
        return False
    return (
        output.get('positivity_ok', 0) == 1 and
        output.get('unitarity_ok', 0) == 1 and
        output.get('perturbativity_ok', 0) == 1
    )

In [21]:
# Coeficientes del polinomio deg=2
a2, a1, a0 = 9.653e-08, 1.301e-06, -9.184e-05

def predict_m12_opt_poly(mphi):
    """Predice m12_opt a partir de mphi usando el polinomio filtrado."""
    return a2*mphi**2 + a1*mphi + a0

In [22]:


# mphi_range = np.linspace(125.05, 299.95, 200)
# m12_est   = predict_m12_opt_poly(mphi_range)




Tecnicas para enconrar un punto que minimice aun mas la funcion de perdida, en esperanzas de hallar los puntos validos cerca de esa minimizacion.

In [23]:
from scipy.optimize import minimize

def refine_m12_around(mphi, mA, span=1e-4):
    m12_guess = predict_m12_opt_poly(mphi)
    def obj(m12):
        return loss(mphi, m12, mA)
    res = minimize(
        obj,
        x0=[m12_guess],
        bounds=[(m12_guess-span, m12_guess+span)],
        method='L-BFGS-B'
    )
    return res.x[0], res.fun

# Ejemplo:
mphi_test, mA_test = 200.0, 300.0
m12_init = predict_m12_opt_poly(mphi_test)
m12_ref, loss_ref = refine_m12_around(mphi_test, mA_test, span=1e-8)
print(f"Inicial: {m12_init:.6e},  Refinado: {m12_ref:.6e},  loss={loss_ref:.6e}")


Inicial: 4.029560e-03,  Refinado: 4.029560e-03,  loss=3.182037e+01


In [24]:
import numpy as np
from scipy.optimize import minimize
def find_valid_m12(mphi, mA,
                   m12_guess,
                   loss_fun,
                   is_valid_fun,
                   bounds=(0.0, 0.02),
                   local_span=1e-5,
                   n_random=200,
                   random_span=1e-4,
                   expand_factor=2.0,
                   max_rounds=5,
                   random_state=None):
    """
    Devuelve (m12_valid, out_dict) o lanza RuntimeError.
    """
    rng = np.random.default_rng(random_state)
    stats = {
        'total_evals': 0,
        'local_eval': 0,
        'random_eval': 0,
        'rounds': 0,
        'success': False,
    }
    
    # 1) Búsqueda local
    def obj(m12):
        stats['local_eval'] += 1
        stats['total_evals'] += 1
        return loss_fun(mphi, m12[0], mA)
    
    res = minimize(
        obj,
        x0=[m12_guess],
        bounds=[(max(bounds[0], m12_guess-local_span),
                 min(bounds[1], m12_guess+local_span))],
        method='L-BFGS-B'
    )
    m12_ref = res.x[0]
    out = physpoint(mphi, m12_ref, mA)
    stats['total_evals'] += 1
    if out is not None and is_valid_fun(out):
        stats['success'] = True
        return m12_ref, out, stats
    
    # 2) Muestreo aleatorio creciente
    span = random_span
    for round_i in range(max_rounds):
        stats['rounds'] += 1
        low = max(bounds[0], m12_ref - span)
        high= min(bounds[1], m12_ref + span)
        m12_cands = rng.uniform(low, high, size=n_random)
        for m12_try in m12_cands:
            out = physpoint(mphi, m12_try, mA)
            stats['random_eval'] += 1
            stats['total_evals'] += 1
            if out is not None and is_valid_fun(out):
                stats['success'] = True
                return m12_try, out, stats
        span *= expand_factor

    raise RuntimeError(f"No encontré punto válido tras {max_rounds} rondas.", stats)


In [ ]:
# nota la funcion tiene pinta de nunca terminar

In [27]:
# ----------------------------
# uso de la funcion
# ----------------------------
mphi_test, mA_test = 200.0, 300.0
m12_guess = predict_m12_opt_poly(mphi_test)  # <==== usa el polinomio para una estimacion base

success_count = 0
total_trials = 10
results = []

for seed in range(total_trials):
    try:
        m12_valid, out_valid, stats = find_valid_m12(
            mphi=mphi_test,
            mA=mA_test,
            m12_guess=m12_guess,
            loss_fun=loss,
            is_valid_fun=is_valid,
            bounds=(0.0, 1.0),
            local_span=1e-1,
            n_random=500,
            random_span=1e-1,
            expand_factor=2.0,
            max_rounds=40,
            random_state=seed
        )
        success_count += 1
        results.append(stats)
    except RuntimeError as err:
        _, stats = err.args
        results.append(stats)

# Estadísticas
avg_evals = np.mean([r['total_evals'] for r in results]) 
avg_rounds = np.mean([r['rounds'] for r in results])
success_rate = success_count / total_trials

print(f"Éxito en {success_count}/{total_trials} ({success_rate*100:.1f}%)")
print(f"Evaluaciones promedio: {avg_evals:.1f}")
print(f"Promedio de rondas aleatorias: {avg_rounds:.1f}")


Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/home/ftrigo/Dihiggs/mlpython/ubenv/lib/python3.8/site-packages/IPython/core/interactiveshell.py", line 3508, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_32859/866487610.py", line 13, in <module>
    m12_valid, out_valid, stats = find_valid_m12(
  File "/tmp/ipykernel_32859/575836479.py", line 54, in find_valid_m12
    out = physpoint(mphi, m12_try, mA)
  File "/tmp/ipykernel_32859/3540951631.py", line 20, in physpoint
    return safe_run_oracle([mphi, mA, sin_ba, tanbeta, lambda6, lambda7, m12_2])
  File "/home/ftrigo/Dihiggs/mlpython/lib/oracle.py", line 89, in safe_run_oracle
    Llama a run_oracle y devuelve salida uniforme con NaNs en caso de error.
  File "/home/ftrigo/Dihiggs/mlpython/lib/oracle.py", line 61, in run_oracle
  File "/usr/lib/python3.8/subprocess.py", line 495, in run
    stdout, stderr = process.communicate(input, timeout=timeout)
  File "/usr/lib/python3.8/subprocess.py", line

In [ ]:
m12_valid, out_valid, stats = find_valid_m12(
            mphi=mphi_test,
            mA=mA_test,
            m12_guess=m12_guess,
            loss_fun=loss,
            is_valid_fun=is_valid,
            bounds=(0.0, 0.01),
            local_span=1e-5,
            n_random=500,
            random_span=1e-4,
            expand_factor=2.0,
            max_rounds=40,
            random_state=seed
        )

print(stats)
print(m12_valid)
out_valid

{'total_evals': 3, 'local_eval': 2, 'random_eval': 0, 'rounds': 0, 'success': True}
0.004029560000000001


{'positivity_ok': 1,
 'unitarity_ok': 1,
 'perturbativity_ok': 1,
 'w_h2_bb': 0.0,
 'w_h2_tautau': 0.0,
 'w_h2_uu': 0.0,
 'w_h2_du': 0.0,
 'w_h2_ln': 0.0,
 'w_h2_vv': [5.933250494227302e-05, 0.208261792487058, 0.59992537348213],
 'w_h2_gaga': 5.933250494227302e-05,
 'w_h2_Zga': 0.0001577370737429793,
 'w_h2_gg': 0.0,
 'w_h2_hh': 0.0,
 'w_total_h2': 0.8084042355478732,
 'w_total_top': 1.338069402459526,
 'branching_ratio_h2_gaga': 7.339459930224399e-05,
 'lambda1': 0.6189673288860724,
 'lambda2': 4.860325572691197,
 'lambda3': 2.338544093321026,
 'lambda4': -1.640323664932234,
 'lambda5': -1.640323664932234,
 'lambda6': 0.1,
 'lambda7': 0.0}

In [ ]:
# un punto conocido de los les houches obtenidos
mphi_test, mA_test = 200, 300
m_12_known = 3.99990900

physpoint(mphi_test, m_12_known, mA_test)

{'positivity_ok': 1,
 'unitarity_ok': 1,
 'perturbativity_ok': 1,
 'w_h2_bb': 0.0,
 'w_h2_tautau': 0.0,
 'w_h2_uu': 0.0,
 'w_h2_du': 0.0,
 'w_h2_ln': 0.0,
 'w_h2_vv': [5.933242715807803e-05, 0.208261792487058, 0.59992537348213],
 'w_h2_gaga': 5.933242715807803e-05,
 'w_h2_Zga': 0.000157736996462813,
 'w_h2_gg': 0.0,
 'w_h2_hh': 0.0,
 'w_total_h2': 0.8084042353928089,
 'w_total_top': 1.338069402459526,
 'branching_ratio_h2_gaga': 7.339450309689189e-05,
 'lambda1': 0.6189439935504256,
 'lambda2': 4.858126952853728,
 'lambda3': 2.338317586076402,
 'lambda4': -1.64009715768761,
 'lambda5': -1.64009715768761,
 'lambda6': 0.1,
 'lambda7': 0.0}